# Regular Expressions (RegEx) for Advanced Filtering

Simple string matching (like `"gmail"`) only works when you know the exact characters you are looking for. But what if you need to find text matching a general *pattern*—like finding all phone numbers, postal codes, or strings that start with a capital letter followed by numbers?

This is where **Regular Expressions (RegEx)** come in. By default, Pandas `.str.contains()` has RegEx enabled. In this module, you will learn:
* How to use RegEx pattern search inside `.str.contains()`.
* How to make searches case-insensitive using `case=False` or RegEx flags.
* How to look for negative/inverted patterns using the bitwise invert operator (`~`).

### Simple Explanation & Real-World Analogy
Think of a standard string match as a "Wanted" poster with an exact photograph of a suspect. You can only find that exact person.
Think of a RegEx pattern match as a description: "Suspect is wearing a red hat, blue jacket, and is between 5.5 and 6 feet tall." Now, you can find *anyone* who matches that description, even if you've never seen them before.

### Code Examples

Let's use a DataFrame containing medical code reports where codes are structured as letters followed by numbers (e.g., `A-12`, `B-45`).


In [1]:
import pandas as pd

data = {
    'Patient': ['John', 'Sarah', 'Emily', 'David', 'Alex'],
    'Medical_Code': ['A-1234', 'b-5678', 'C-99', 'INVALID_CODE', 'A-7788']
}
df = pd.DataFrame(data)
print(df)

  Patient  Medical_Code
0    John        A-1234
1   Sarah        b-5678
2   Emily          C-99
3   David  INVALID_CODE
4    Alex        A-7788


#### Basic RegEx Filtering
Let's filter rows where the `Medical_Code` follows the pattern of **"One letter, a hyphen, and some digits"**.
* In RegEx, `[a-zA-Z]` matches any single English letter.
* `-` matches a literal hyphen.
* `\d+` matches one or more digits.

In [2]:
# Filter codes that match: Letter-Digits
pattern = r'[a-zA-Z]-\d+'
matched_df = df[df['Medical_Code'].str.contains(pattern, na=False)]
print(matched_df)

  Patient Medical_Code
0    John       A-1234
1   Sarah       b-5678
2   Emily         C-99
4    Alex       A-7788



#### Case-Insensitivity in RegEx
Suppose we only want medical codes that start with the letter **"A"** or **"B"** (case-insensitive).
We can use `case=False` inside `.str.contains()`.

In [3]:
# Search for codes starting with A or B, ignoring uppercase/lowercase differences
ab_codes = df[df['Medical_Code'].str.contains('^[ab]-', case=False, na=False)]
print(ab_codes)

  Patient Medical_Code
0    John       A-1234
1   Sarah       b-5678
4    Alex       A-7788


*(Note: `^` in RegEx means "starts with".)*

#### Inverting the Filter (`~`)
If we want to find all rows that do **NOT** have a valid medical code pattern, we can place the tilde (`~`) operator in front of our boolean mask.

In [4]:
# Invert the mask to find invalid records
invalid_records = df[~df['Medical_Code'].str.contains(r'[a-zA-Z]-\d+', na=False)]
print(invalid_records)

  Patient  Medical_Code
3   David  INVALID_CODE


### Common Mistakes Beginners Make
1. **Misunderstanding `^` (Starts with) vs `$` (Ends with):**
   If you search for `r'\d+'`, it matches any string containing digits *anywhere*. If you want a string that contains *only* digits, you must anchor it: `r'^\d+$'`.
2. **Forgetting to disable regex when doing literal searches:**
   If you want to find a literal question mark `?` or period `.`, you must escape it with a backslash `\?` or `\.` because these are special RegEx characters. Alternatively, set `regex=False` inside `.str.contains('?', regex=False)`.


#### Exercise 1 (Medium)
You have a DataFrame of user registrations with email addresses:
```python
users = pd.DataFrame({
    'User': ['User1', 'User2', 'User3', 'User4'],
    'Email': ['test.user@gmail.com', 'admin@company.org', 'support@edu.net', 'bad_email.com']
})
```
Write a RegEx pattern to filter and return only the users who have a valid email domain containing `.com` or `.org`. (Hint: Use `com|org`).


In [6]:
import pandas as pd

users = pd.DataFrame({
    'User': ['User1', 'User2', 'User3', 'User4'],
    'Email': ['test.user@gmail.com', 'admin@company.org', 'support@edu.net', 'bad_email.com']
})

# Pattern matches a literal '.' followed by 'com' or 'org' at the end of the string ($).
# (?: ... ) is a NON-CAPTURING group: it groups the alternatives 'com|org' together
# but does not save them as a match group. Plain ( ... ) would save them, and
# .str.contains() then warns that we should use .str.extract() to read the groups.
# We only want True/False here, so the non-capturing version is the right one.
pattern = r'\.(?:com|org)$'
valid_users = users[users['Email'].str.contains(pattern, na=False)]
print(valid_users)

    User                Email
0  User1  test.user@gmail.com
1  User2    admin@company.org
3  User4        bad_email.com
